## audit を日別×ユーザーで集計

**スキーマ例**

```
event_date DATE
user_email STRING
audit_event_count LONG
distinct_resource_count LONG
get_table_count LONG
command_submit_count LONG
```

In [0]:
%run ./config

In [0]:
gold_table_name = "audit_user_daily"
gold_table_path = f"{catalog_name}.{schema_name}.{gold_table_name}"


In [0]:
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {gold_table_path} (
        event_date DATE,
        user_email STRING,
        audit_event_count LONG,
        distinct_resource_count LONG,
        get_table_count LONG,
        command_submit_count LONG
    )
    PARTITIONED BY (event_date)
    """
)

In [0]:
# 日別ユーザー集計
df = spark.sql(
    f"""
    SELECT
        CAST(event_time AS DATE) AS event_date,
        user_email,
        COUNT(*) AS audit_event_count,
        COUNT(DISTINCT resource_name) AS distinct_resource_count,
        SUM(CASE WHEN action_name = 'getTable' THEN 1 ELSE 0 END) AS get_table_count,
        SUM(CASE WHEN action_name = 'commandSubmit' THEN 1 ELSE 0 END) AS command_submit_count
    FROM {silver_audit_table_path}
    GROUP BY CAST(event_time AS DATE), user_email
    ORDER BY event_date, user_email
    """
)

In [0]:
# テーブルとして保存
df.write.mode("overwrite").saveAsTable(gold_table_path)

print("✅ gold_daily_stats テーブルを作成しました")

In [0]:
# 結果を確認
display(spark.table(gold_table_path))
